In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

cols = ["unit", "cycle", "op1", "op2","op3","s1", "s2", "s3", "s4", "s5", "s6", "s7", "s8", "s9", "s10", "s11", "s12", "s13", "s14", "s15", "s16", "s17", "s18", "s19", "s20", "s21"]

train = pd.read_csv("../data/raw/train_FD001.txt", sep = r"\s+", header = None, names = cols)
test = pd.read_csv("../data/raw/test_FD001.txt", sep = r"\s+", header = None, names = cols)
rul = pd.read_csv("../data/raw/RUL_FD001.txt", sep = r"\s+", header = None, names = ["RUL"])
rul.index = range(1,len(rul)+1)

aus eda_01:

In [14]:
drop_cols = ["op1", "op2","op3", "s1", "s5", "s6", "s10", "s16", "s18", "s19"]

In [15]:
train["max_cycle"] = train.groupby("unit")["cycle"].transform("max")
train["RUL"] = train["max_cycle"] - train["cycle"]

In [16]:
up = ["s2", "s3", "s4", "s8", "s11", "s13", "s15", "s17"]
down= ["s7", "s12", "s20", "s21"]
ambiguous=["s9", "s14"]

features = up + down + ambiguous
len(features)

14

In [17]:
train["RUL_clip"] = train["RUL"].clip(upper=125)
X_train = train[features]
y_train = train["RUL_clip"]
X_test = test.groupby("unit")[features].last()
y_test = rul["RUL"]

In [18]:
X_test.shape

(100, 14)

In [19]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [20]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

np.sqrt(mean_squared_error(y_test, y_pred))

np.float64(21.900213406890536)

In [21]:
from xgboost import XGBRegressor
xgb_model = XGBRegressor(random_state=42)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

In [24]:
np.sqrt(mean_squared_error(y_test, y_pred_xgb))

np.float64(18.177173527563955)

In [25]:
y_pred_train_xgb = xgb_model.predict(X_train)
np.sqrt(mean_squared_error(y_train, y_pred_train_xgb))

np.float64(12.391884738531378)

Anzeichen für Overfitting da RMSE der Testdaten bei 12,4 und bei den Testdaten bei 18,2 also ca 50% höher. Aufgrund unteschiedlicher Messdaten allerdings nicht vergleichbar 